#### Copyright 2018 Google LLC.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Recommendation Systems with TensorFlow

This notebook, adapted from a lab of [Google's course on recommendation systems](https://developers.google.com/machine-learning/recommendation/), learns user and movie embeddings by matrix factorization, on the MovieLens ratings.

# Introduction

We will create a movie recommendation system based on the [MovieLens](https://movielens.org/) dataset available [here](http://grouplens.org/datasets/movielens/).  The data consists of movies ratings (on a scale of 1 to 5).

## Outline
  1. Exploring the MovieLens Data (10 minutes)
  1. Preliminaries (25 minutes)
  1. Training a matrix factorization model (15 minutes)
  1. Inspecting the Embeddings (15 minutes)
  1. Regularization in matrix factorization (15 minutes)
  1. Softmax model training (30 minutes)

## Setup

Let's get started by importing the required packages.

In [ ]:
# @title Imports (run this cell)

import numpy as np
import pandas as pd
import collections
from mpl_toolkits.mplot3d import Axes3D
from IPython import display
from matplotlib import pyplot as plt
import sklearn
import sklearn.manifold
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()
tf.logging.set_verbosity(tf.logging.ERROR)

# Add some convenience functions to Pandas DataFrame.
pd.options.display.max_rows = 10
pd.options.display.float_format = "{:.3f}".format


def mask(df, key, function):
  """Returns a filtered dataframe, by applying function to key"""
  return df[function(df[key])]


def flatten_cols(df):
  df.columns = [" ".join(col).strip() for col in df.columns.values]
  return df


pd.DataFrame.mask = mask
pd.DataFrame.flatten_cols = flatten_cols

# Altair, already installed on Colab, draws the interactive charts.
import altair as alt

alt.data_transformers.enable("default", max_rows=None)

We then download the MovieLens Data, and create DataFrames containing movies, users, and ratings.

In [ ]:
# @title Load the MovieLens data (run this cell).

# Download MovieLens data.
print("Downloading movielens data...")
from urllib.request import urlretrieve
import zipfile

urlretrieve(
  "http://files.grouplens.org/datasets/movielens/ml-100k.zip", "movielens.zip"
)
zip_ref = zipfile.ZipFile("movielens.zip", "r")
zip_ref.extractall()
print("Done. Dataset contains:")
print(zip_ref.read("ml-100k/u.info"))

# Load each data set (users, movies, and ratings).
users_cols = ["user_id", "age", "sex", "occupation", "zip_code"]
users = pd.read_csv("ml-100k/u.user", sep="|", names=users_cols, encoding="latin-1")

ratings_cols = ["user_id", "movie_id", "rating", "unix_timestamp"]
ratings = pd.read_csv(
  "ml-100k/u.data", sep="\t", names=ratings_cols, encoding="latin-1"
)

# The movies file contains a binary feature for each genre.
genre_cols = [
  "genre_unknown",
  "Action",
  "Adventure",
  "Animation",
  "Children",
  "Comedy",
  "Crime",
  "Documentary",
  "Drama",
  "Fantasy",
  "Film-Noir",
  "Horror",
  "Musical",
  "Mystery",
  "Romance",
  "Sci-Fi",
  "Thriller",
  "War",
  "Western",
]
movies_cols = [
  "movie_id",
  "title",
  "release_date",
  "video_release_date",
  "imdb_url",
] + genre_cols
movies = pd.read_csv("ml-100k/u.item", sep="|", names=movies_cols, encoding="latin-1")

# Since the ids start at 1, we shift them to start at 0.
users["user_id"] = users["user_id"].apply(lambda x: str(x - 1))
movies["movie_id"] = movies["movie_id"].apply(lambda x: str(x - 1))
movies["year"] = movies["release_date"].apply(lambda x: str(x).split("-")[-1])
ratings["movie_id"] = ratings["movie_id"].apply(lambda x: str(x - 1))
ratings["user_id"] = ratings["user_id"].apply(lambda x: str(x - 1))
ratings["rating"] = ratings["rating"].apply(lambda x: float(x))

# Compute the number of movies to which a genre is assigned.
genre_occurences = movies[genre_cols].sum().to_dict()


# Since some movies can belong to more than one genre, we create different
# 'genre' columns as follows:
# - all_genres: all the active genres of the movie.
# - genre: randomly sampled from the active genres.
def mark_genres(movies, genres):
  def get_random_genre(gs):
    active = [genre for genre, g in zip(genres, gs) if g == 1]
    if len(active) == 0:
      return "Other"
    return np.random.choice(active)

  def get_all_genres(gs):
    active = [genre for genre, g in zip(genres, gs) if g == 1]
    if len(active) == 0:
      return "Other"
    return "-".join(active)

  movies["genre"] = [
    get_random_genre(gs) for gs in zip(*[movies[genre] for genre in genres])
  ]
  movies["all_genres"] = [
    get_all_genres(gs) for gs in zip(*[movies[genre] for genre in genres])
  ]


mark_genres(movies, genre_cols)

# Create one merged DataFrame containing all the movielens data.
movielens = ratings.merge(movies, on="movie_id").merge(users, on="user_id")


# Utility to split the data into training and test sets.
def split_dataframe(df, holdout_fraction=0.1):
  """Splits a DataFrame into training and test sets.
  Args:
    df: a dataframe.
    holdout_fraction: fraction of dataframe rows to use in the test set.
  Returns:
    train: dataframe for training
    test: dataframe for testing
  """
  test = df.sample(frac=holdout_fraction, replace=False)
  train = df[~df.index.isin(test.index)]
  return train, test

# I. Exploring the Movielens Data
Before we dive into model building, let's inspect our MovieLens dataset. It is usually helpful to understand the statistics of the dataset.

### Users
We start by printing some basic statistics describing the numeric user features.

In [ ]:
users.describe()

We can also print some basic statistics describing the categorical user features

In [ ]:
users.describe(include=[object])

We can also create histograms to further understand the distribution of the users. We use Altair to create an interactive chart.

In [ ]:
# @title Altair visualization code (run this cell)
# The following functions are used to generate interactive Altair charts.
# We will display histograms of the data, sliced by a given attribute.

# Create filters to be used to slice the data.
occupation_filter = alt.selection_point(fields=["occupation"])
occupation_chart = (
  alt.Chart()
  .mark_bar()
  .encode(
    x="count()",
    y=alt.Y("occupation:N"),
    color=alt.condition(
      occupation_filter,
      alt.Color("occupation:N", scale=alt.Scale(scheme="category20")),
      alt.value("lightgray"),
    ),
  )
  .properties(width=300, height=300)
  .add_params(occupation_filter)
)


# A function that generates a histogram of filtered data.
def filtered_hist(field, label, filter):
  """Creates a layered chart of histograms.
  The first layer (light gray) contains the histogram of the full data, and the
  second contains the histogram of the filtered data.
  Args:
    field: the field for which to generate the histogram.
    label: String label of the histogram.
    filter: an alt.Selection object to be used to filter the data.
  """
  base = (
    alt.Chart()
    .mark_bar()
    .encode(
      x=alt.X(field, bin=alt.Bin(maxbins=10), title=label),
      y="count()",
    )
    .properties(
      width=300,
    )
  )
  return alt.layer(
    base.transform_filter(filter),
    base.encode(color=alt.value("lightgray"), opacity=alt.value(0.7)),
  ).resolve_scale(y="independent")

Next, we look at the distribution of ratings per user. Clicking on an occupation in the right chart will filter the data by that occupation. The corresponding histogram is shown in blue, and superimposed with the histogram for the whole data (in light gray). You can use SHIFT+click to select multiple subsets.

What do you observe, and how might this affect the recommendations?

In [ ]:
users_ratings = (
  ratings.groupby("user_id", as_index=False)
  .agg({"rating": ["count", "mean"]})
  .flatten_cols()
  .merge(users, on="user_id")
)

# Create a chart for the count, and one for the mean.
alt.hconcat(
  filtered_hist("rating count", "# ratings / user", occupation_filter),
  filtered_hist("rating mean", "mean user rating", occupation_filter),
  occupation_chart,
  data=users_ratings,
)

### Movies

It is also useful to look at information about the movies and their ratings.

In [ ]:
movies_ratings = movies.merge(
  ratings.groupby("movie_id", as_index=False)
  .agg({"rating": ["count", "mean"]})
  .flatten_cols(),
  on="movie_id",
)

genre_filter = alt.selection_point(fields=["genre"])
genre_chart = (
  alt.Chart()
  .mark_bar()
  .encode(
    x="count()",
    y=alt.Y("genre"),
    color=alt.condition(genre_filter, alt.Color("genre:N"), alt.value("lightgray")),
  )
  .properties(height=300)
  .add_params(genre_filter)
)

In [ ]:
(
  movies_ratings[["title", "rating count", "rating mean"]]
  .sort_values("rating count", ascending=False)
  .head(10)
)

In [ ]:
(
  movies_ratings[["title", "rating count", "rating mean"]]
  .mask("rating count", lambda x: x > 20)
  .sort_values("rating mean", ascending=False)
  .head(10)
)

Finally, the last chart shows the distribution of the number of ratings and average rating.

In [ ]:
# Display the number of ratings and average rating per movie.
alt.hconcat(
  filtered_hist("rating count", "# ratings / movie", genre_filter),
  filtered_hist("rating mean", "mean movie rating", genre_filter),
  genre_chart,
  data=movies_ratings,
)

# II. Preliminaries

Our goal is to factorize the ratings matrix $M$ (users in rows, movies in columns) into the product of a user embedding matrix $U$ and a movie embedding matrix $V$, such that $M \approx UV^\top$ with
$U = \begin{bmatrix} u_{1} \\ \hline \vdots \\ \hline u_{n} \end{bmatrix}$ and
$V = \begin{bmatrix} v_{1} \\ \hline \vdots \\ \hline v_{m} \end{bmatrix}$.

Here
- $n$ is the number of users,
- $m$ is the number of movies,
- $M_{ij}$ is the rating of the $j$th movie by the $i$th user,
- each row $u_i$ of $U$ is a $d$-dimensional vector (embedding) representing user $i$,
- each row $v_j$ of $V$ is a $d$-dimensional vector (embedding) representing movie $j$,
- the prediction of the model for the $(i, j)$ pair is the dot product of their embeddings: $M_{ij} \approx u_i \cdot v_j$.

## Sparse Representation of the Rating Matrix

The rating matrix could be very large and, in general, most of the entries are unobserved, since a given user will only rate a small subset of movies. For efficient representation, we will use a [tf.SparseTensor](https://www.tensorflow.org/api_docs/python/tf/sparse/SparseTensor). A `SparseTensor` uses three tensors to represent the matrix: `tf.SparseTensor(indices, values, dense_shape)` represents a tensor, where a value $M_{ij} = a$ is encoded by setting `indices[k] = [i, j]` and `values[k] = a`. The last tensor `dense_shape` is used to specify the shape of the full underlying matrix.

#### Toy example
Assume we have $2$ users and $4$ movies. Our toy ratings dataframe has three ratings,

user\_id | movie\_id | rating
--:|--:|--:
0 | 0 | 5.0
0 | 1 | 3.0
1 | 3 | 1.0

The corresponding rating matrix is

$$
M =
\begin{bmatrix}
5.0 & 3.0 & 0 & 0 \\
0   &   0 & 0 & 1.0
\end{bmatrix}
$$

And the SparseTensor representation is,
```python
SparseTensor(
  indices=[[0, 0], [0, 1], [1,3]],
  values=[5.0, 3.0, 1.0],
  dense_shape=[2, 4])
```

### Exercise 1: Build a tf.SparseTensor representation of the Rating Matrix

Write the `build_rating_sparse_tensor` function, which maps a ratings DataFrame (columns `user_id`, `movie_id` and `rating`) to a `tf.SparseTensor` of shape `dense_shape`: by default, the shape of the MovieLens ratings matrix, `users.shape[0]` users and `movies.shape[0]` movies.

Then check it on the toy example: build the sparse matrix of `toy_ratings` (defined in the next cell), of shape `[2, 4]`, convert it to a dense matrix with `tf.sparse.to_dense`, and compare the result with the matrix $M$ above. This notebook uses TensorFlow in graph mode (the TensorFlow 1 API): a tensor only gets a value when computed by the `run` method of a `tf.Session`.

**Useful documentation**

- [`tf.sparse.SparseTensor`](https://www.tensorflow.org/api_docs/python/tf/sparse/SparseTensor)
- [`tf.sparse.to_dense`](https://www.tensorflow.org/api_docs/python/tf/sparse/to_dense)
- [`tf.compat.v1.Session`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/Session)

In [ ]:
toy_ratings = pd.DataFrame(
  {"user_id": ["0", "0", "1"], "movie_id": ["0", "1", "3"], "rating": [5.0, 3.0, 1.0]}
)


def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
    dense_shape: the shape [n, m] of the ratings matrix; by default, the number
      of users and the number of movies of MovieLens.
  Returns:
    A tf.SparseTensor representing the ratings matrix.
  """
  # Your code here
  ...


# Your code here: the check on toy_ratings

#### Additional hints

The calls and parameters to use:

- `ratings_df[["user_id", "movie_id"]].values`: an array of shape (number of ratings, 2), the `[i, j]` pair of each rating (the ids are strings, `"0"`, `"3"`, which TensorFlow converts to integers)
- `ratings_df["rating"].values`: the ratings, in the same order
- `tf.SparseTensor(indices=..., values=..., dense_shape=...)`
- `[users.shape[0], movies.shape[0]]`: the default shape, when `dense_shape` is `None`
- `with tf.Session() as sess:`, then `sess.run(tensor)`: the value of a tensor, a NumPy array

On the toy example, `tf.sparse.to_dense` must give back `[[5. 3. 0. 0.] [0. 0. 0. 1.]]`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  if dense_shape is None:
    dense_shape = [users.shape[0], movies.shape[0]]
  indices = ratings_df[...].values
  values = ...
  return tf.SparseTensor(indices=indices, values=values, dense_shape=dense_shape)


toy_sparse = build_rating_sparse_tensor(toy_ratings, dense_shape=...)
with tf.Session() as sess:
  print(sess.run(tf.sparse.to_dense(...)))
```

#### Solution

In [ ]:
def build_rating_sparse_tensor(ratings_df, dense_shape=None):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
    dense_shape: the shape [n, m] of the ratings matrix; by default, the number
      of users and the number of movies of MovieLens.
  Returns:
    A tf.SparseTensor representing the ratings matrix.
  """
  if dense_shape is None:
    dense_shape = [users.shape[0], movies.shape[0]]
  indices = ratings_df[["user_id", "movie_id"]].values
  values = ratings_df["rating"].values
  return tf.SparseTensor(indices=indices, values=values, dense_shape=dense_shape)


toy_sparse = build_rating_sparse_tensor(toy_ratings, dense_shape=[2, 4])
with tf.Session() as sess:
  print(sess.run(tf.sparse.to_dense(toy_sparse)))

## Calculating the error

The model approximates the ratings matrix $M$ by a low-rank product $UV^\top$. We need a way to measure the approximation error. We'll start by using the Mean Squared Error of observed entries only (we will revisit this later). It is defined as

$$
\begin{align*}
\text{MSE}(M, UV^\top)
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( M_{ij} - (UV^\top)_{ij})^2} \\
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( M_{ij} - u_i \cdot v_j)^2}
\end{align*}
$$
where $\Omega$ is the set of observed ratings, and $|\Omega|$ is the cardinality of $\Omega$.

### Exercise 2: Mean Squared Error

Write a TensorFlow function that takes a sparse rating matrix $M$ and the two embedding matrices $U, V$ and returns the mean squared error $\text{MSE}(M, UV^\top)$, computed on the observed ratings only. Check it on the toy example, `toy_sparse`, with the embeddings of the hints.

**Useful documentation**

- [`tf.linalg.matmul`](https://www.tensorflow.org/api_docs/python/tf/linalg/matmul)
- [`tf.gather_nd`](https://www.tensorflow.org/api_docs/python/tf/gather_nd)
- [`tf.compat.v1.losses.mean_squared_error`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/losses/mean_squared_error)

In [ ]:
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  # Your code here
  ...


# Your code here: the check on toy_sparse

#### Additional hints

The calls and parameters to use:

- in this section, we only consider observed entries when calculating the loss
- `sparse_ratings.indices`, `sparse_ratings.values`: the observed $(i, j)$ pairs and their ratings
- `tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)`: the matrix $UV^\top$ of all the predictions
- `tf.gather_nd(matrix, indices)`: the entries of `matrix` at the `indices` pairs
- `tf.losses.mean_squared_error(ratings, predictions)`: the mean of the squared differences

On the toy example, with $U = \begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}$ and $V = \begin{bmatrix} 2 & 2 \\ 1 & 1 \\ 0 & 0 \\ 0 & 1 \end{bmatrix}$, the predictions of the three observed ratings (5, 3 and 1) are 4, 2 and 0: the MSE is 1.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  predictions = tf.gather_nd(tf.matmul(..., ..., transpose_b=True), ...)
  return tf.losses.mean_squared_error(..., predictions)


toy_U = tf.constant([[1.0, 1.0], [1.0, 0.0]])
toy_V = tf.constant([[2.0, 2.0], [1.0, 1.0], [0.0, 0.0], [0.0, 1.0]])
with tf.Session() as sess:
  print(sess.run(sparse_mean_square_error(toy_sparse, toy_U, toy_V)))
```

#### Solution

In [ ]:
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.gather_nd(
    tf.matmul(user_embeddings, movie_embeddings, transpose_b=True),
    sparse_ratings.indices,
  )
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss


toy_U = tf.constant([[1.0, 1.0], [1.0, 0.0]])
toy_V = tf.constant([[2.0, 2.0], [1.0, 1.0], [0.0, 0.0], [0.0, 1.0]])
with tf.Session() as sess:
  print(sess.run(sparse_mean_square_error(toy_sparse, toy_U, toy_V)))

Note: One approach is to compute the full prediction matrix $UV^\top$, then gather the entries corresponding to the observed pairs. The memory cost of this approach is $O(nm)$. For the MovieLens dataset, this is fine, as the dense $n \times m$ matrix is small enough to fit in memory ($n = 943$, $m = 1682$).

Another approach (given in the alternate solution below) is to only gather the embeddings of the observed pairs, then compute their dot products. The memory cost is $O(|\Omega| d)$ where $d$ is the embedding dimension. In our case, $|\Omega| = 10^5$, and the embedding dimension is on the order of $10$, so the memory cost of both methods is comparable. But when the number of users or movies is much larger, the first approach becomes infeasible.

In [ ]:
# Alternate solution
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix M, of dense_shape [n, m].
    user_embeddings: A dense Tensor U of shape [n, d] where d is the embedding
      dimension, such that row i, u_i, is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [m, d] where d is the embedding
      dimension, such that row j, v_j, is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.reduce_sum(
    tf.gather(user_embeddings, sparse_ratings.indices[:, 0])
    * tf.gather(movie_embeddings, sparse_ratings.indices[:, 1]),
    axis=1,
  )
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss

### Exercise 3 (Optional): adding your own ratings to the data set

You can add your own ratings to the data set, to then get recommendations for yourself (section IV).

Enter your ratings in the `my_ratings` dictionary of the next cell: for each movie title, written as in `movies["title"]`, a rating from 1 to 5; the ratings it holds are examples, to be replaced by yours. Then, in the same cell, add these ratings to the `ratings` DataFrame with `pd.concat`, under the user id `"943"` (MovieLens has 943 of them, from `"0"` to `"942"`), and add a row to `users` for this new user, so that the ratings matrix gets one more row. Running the cell a second time must replace your ratings, not add them a second time.

**Useful documentation**

- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
# Your ratings: a title from movies["title"], a rating from 1 to 5.
my_ratings = {
  "Star Wars (1977)": 5,
  "Toy Story (1995)": 4,
  "Aladdin (1992)": 4,
  "Pulp Fiction (1994)": 2,
  "Fargo (1996)": 1,
}

# Your code here

#### Additional hints

The calls and parameters to use:

- `dict(zip(movies["title"], movies["movie_id"]))`: a dictionary giving the id (`movie_id`) of each title
- `pd.DataFrame({"user_id": "943", "movie_id": [...], "rating": [...]})`: the DataFrame of your ratings; a single value, like `"943"`, is repeated on every row
- `ratings[ratings["user_id"] != "943"]`: the ratings, without those of user `"943"` (so that a second run replaces your ratings)
- `pd.concat([df1, df2], ignore_index=True)`: the rows of `df1` then those of `df2`, renumbered
- a misspelled title raises a `KeyError`: `movies[movies["title"].str.contains("Star Wars")]` finds the exact title

With the example ratings, `ratings` goes from 100,000 to 100,005 rows, and `users` from 943 to 944.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
title_to_id = dict(zip(movies["title"], movies["movie_id"]))
my_ratings_df = pd.DataFrame(
  {
    "user_id": "943",
    "movie_id": [title_to_id[title] for title in ...],
    "rating": [float(rating) for rating in ...],
  }
)
ratings = ratings[ratings["user_id"] != "943"]
ratings = pd.concat([..., ...], ignore_index=True)
if users.shape[0] == 943:
  users = pd.concat([users, pd.DataFrame({"user_id": ["943"]})], ignore_index=True)
```

#### Solution

In [ ]:
title_to_id = dict(zip(movies["title"], movies["movie_id"]))
my_ratings_df = pd.DataFrame(
  {
    "user_id": "943",
    "movie_id": [title_to_id[title] for title in my_ratings],
    "rating": [float(rating) for rating in my_ratings.values()],
  }
)
# Remove the previous ratings of user 943, then add the new ones.
ratings = ratings[ratings["user_id"] != "943"]
ratings = pd.concat([ratings, my_ratings_df], ignore_index=True)
# Add the new user to the users DataFrame.
if users.shape[0] == 943:
  users = pd.concat([users, pd.DataFrame({"user_id": ["943"]})], ignore_index=True)
print("Successfully added %d ratings; you have great taste!" % len(my_ratings_df))
ratings[ratings["user_id"] == "943"].merge(movies[["movie_id", "title"]])

# III. Training a Matrix Factorization model

## CFModel (Collaborative Filtering Model) helper class
This is a simple class to train a matrix factorization model using gradient descent.

The class constructor takes
- a dictionary of the embedding variables (`tf.Variable`s), for example `{"user_id": U, "movie_id": V}`;
- a loss to optimize (a `tf.Tensor`);
- an optional list of metrics dictionaries, each mapping a string (the name of the metric) to a tensor. These are evaluated during training, and each dictionary is plotted in its own chart at the end (e.g. training error and test error).

`model.train` returns the values of the metrics at the last iteration. After training, one can access the trained embeddings using the `model.embeddings` dictionary.

Example usage:
```
U = ...
V = ...
loss = ...
model = CFModel({"user_id": U, "movie_id": V}, loss, [{"train_error": loss}])
model.train(num_iterations=100, learning_rate=1.0)
user_embeddings = model.embeddings["user_id"]
movie_embeddings = model.embeddings["movie_id"]
```

In [ ]:
# @title CFModel helper class (run this cell)
class CFModel(object):
  """Simple class that represents a collaborative filtering model"""

  def __init__(self, embedding_vars, loss, metrics=None):
    """Initializes a CFModel.
    Args:
      embedding_vars: A dictionary of tf.Variables.
      loss: A float Tensor. The loss to optimize.
      metrics: optional list of dictionaries of Tensors. The metrics in each
        dictionary will be plotted in a separate figure during training.
    """
    self._embedding_vars = embedding_vars
    self._loss = loss
    self._metrics = metrics
    self._embeddings = {k: None for k in embedding_vars}
    self._session = None

  @property
  def embeddings(self):
    """The embeddings dictionary."""
    return self._embeddings

  def train(
    self,
    num_iterations=100,
    learning_rate=1.0,
    plot_results=True,
    optimizer=tf.train.GradientDescentOptimizer,
  ):
    """Trains the model.
    Args:
      num_iterations: number of iterations to run.
      learning_rate: optimizer learning rate.
      plot_results: whether to plot the results at the end of training.
      optimizer: the optimizer to use. Default to GradientDescentOptimizer.
    Returns:
      The metrics dictionary evaluated at the last iteration.
    """
    with self._loss.graph.as_default():
      opt = optimizer(learning_rate)
      train_op = opt.minimize(self._loss)
      local_init_op = tf.group(
        tf.variables_initializer(opt.variables()), tf.local_variables_initializer()
      )
      if self._session is None:
        self._session = tf.Session()
        with self._session.as_default():
          self._session.run(tf.global_variables_initializer())
          self._session.run(tf.tables_initializer())
          tf.train.start_queue_runners()

    with self._session.as_default():
      local_init_op.run()
      iterations = []
      metrics = self._metrics or ({},)
      metrics_vals = [collections.defaultdict(list) for _ in self._metrics]

      # Train and append results.
      for i in range(num_iterations + 1):
        _, results = self._session.run((train_op, metrics))
        if (i % 10 == 0) or i == num_iterations:
          print(
            "\r iteration %d: " % i
            + ", ".join(["%s=%f" % (k, v) for r in results for k, v in r.items()]),
            end="",
          )
          iterations.append(i)
          for metric_val, result in zip(metrics_vals, results):
            for k, v in result.items():
              metric_val[k].append(v)

      for k, v in self._embedding_vars.items():
        self._embeddings[k] = v.eval()

      if plot_results:
        # Plot the metrics.
        num_subplots = len(metrics)
        fig = plt.figure()
        fig.set_size_inches(num_subplots * 10, 8)
        for i, metric_vals in enumerate(metrics_vals):
          ax = fig.add_subplot(1, num_subplots, i + 1)
          for k, v in metric_vals.items():
            ax.plot(iterations, v, label=k)
          ax.set_xlim([1, num_iterations])
          ax.legend()
      return results

## Building the model

The `build_model` function of the next cell builds a `CFModel`: it splits the ratings into training and test sets, represents them as `SparseTensor`s with your `build_rating_sparse_tensor` function, initializes the embeddings $U$ and $V$ from a normal distribution of standard deviation `init_stddev`, and defines the training and test errors with your `sparse_mean_square_error` function.

In [ ]:
# @title Building the model (run this cell)
def build_model(ratings, embedding_dim=3, init_stddev=1.0):
  """
  Args:
    ratings: a DataFrame of the ratings
    embedding_dim: the dimension of the embedding vectors.
    init_stddev: float, the standard deviation of the random initial embeddings.
  Returns:
    model: a CFModel.
  """
  # Split the ratings DataFrame into train and test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # SparseTensor representation of the train and test datasets.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  # Initialize the embeddings using a normal distribution.
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )
  train_loss = sparse_mean_square_error(A_train, U, V)
  test_loss = sparse_mean_square_error(A_test, U, V)
  metrics = {"train_error": train_loss, "test_error": test_loss}
  embeddings = {"user_id": U, "movie_id": V}
  return CFModel(embeddings, train_loss, [metrics])

Great, now it's time to train the model!

Go ahead and run the next cell, trying different parameters (embedding dimension, learning rate, iterations). The training and test errors are plotted at the end of training. You can inspect these values to validate the hyper-parameters.

Note: by calling `model.train` again, the model will continue training starting from the current values of the embeddings.

In [ ]:
# Build the CF model and train it.
model = build_model(ratings, embedding_dim=30, init_stddev=0.5)
model.train(num_iterations=1000, learning_rate=10.0)

The chart plots the training and test errors over the iterations. Section IV will have a more detailed look at the learned embeddings.

### Exercise 4: Embedding dimension and overfitting

Train three models with `build_model`, varying `embedding_dim`: 3, 30 and 100, with `init_stddev=0.5`, `learning_rate=10.0` and 300 iterations each (fewer than the 1,000 of the previous cell, so that the exercise takes about a minute). For each one, note the training and test errors of the last iteration.

As the dimension grows, what happens to the training error, the test error and their gap? From which dimension does the model overfit?

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `build_model(ratings, embedding_dim=..., init_stddev=0.5)`: a new model; give it another name than `model`, which the rest of the notebook uses
- `.train(num_iterations=300, learning_rate=10.0, plot_results=False)`: the training, without a chart; it returns a list of one dictionary, `{"train_error": ..., "test_error": ...}`, the errors of the last iteration
- `train` prints its progress without a newline: a `print()` before yours

The values change from one run to the next (the split and the initialization are random), but not their order of magnitude: about 0.85 for training and 1 for test in dimension 3, 0.3 and 3.3 in dimension 100.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
for embedding_dim in [3, 30, 100]:
  dim_model = build_model(ratings, embedding_dim=..., init_stddev=0.5)
  results = dim_model.train(num_iterations=300, learning_rate=10.0, plot_results=False)
  errors = results[0]
  print()
  print(embedding_dim, errors[...], errors[...])
```

#### Solution

In [ ]:
for embedding_dim in [3, 30, 100]:
  dim_model = build_model(ratings, embedding_dim=embedding_dim, init_stddev=0.5)
  results = dim_model.train(num_iterations=300, learning_rate=10.0, plot_results=False)
  errors = results[0]
  gap = errors["test_error"] - errors["train_error"]
  values = (embedding_dim, errors["train_error"], errors["test_error"], gap)
  print("\nDimension %d: training %.2f, test %.2f, gap %.2f" % values)

- As the dimension grows, the training error goes down and the test error goes up: the gap goes from about 0.1 in dimension 3 to more than 0.5 in dimension 30, and to about 3 in dimension 100.
- The model overfits from dimension 30 on: a user has about a hundred ratings on average, 90% of them in training, and an embedding of 30 or 100 coordinates has enough parameters to learn them by heart, without learning anything about their test ratings.
- With dimension 3, the model generalizes better, but its embeddings are too poor to be worth inspecting: section V keeps a large dimension and adds regularization.

# IV. Inspecting the Embeddings

In this section, we take a closer look at the learned embeddings, by
- computing your recommendations
- looking at the nearest neighbors of some movies,
- looking at the norms of the movie embeddings,
- visualizing the embedding in a projected embedding space.

### Exercise 5: Write a function that computes the scores of the candidates
We start by writing a function that, given a query embedding $u \in \mathbb R^d$ and item embeddings $V \in \mathbb R^{m \times d}$ (one row $v_j$ per item), computes the item scores.

As discussed in the lecture, there are different similarity measures we can use, and these can yield different results. We will compare the following:
- dot product: the score of item $j$ is $u \cdot v_j$;
- cosine: the score of item $j$ is $\frac{u \cdot v_j}{\|u\|\|v_j\|}$.

**Useful documentation**

- [`numpy.dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

In [ ]:
DOT = "dot"
COSINE = "cosine"


def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  # Your code here
  ...

#### Additional hints

The calls and parameters to use:

- `u.dot(V.T)`: the vector of the dot products $u \cdot v_j$, one per item
- `np.linalg.norm(V, axis=1, keepdims=True)`: the norm of each row of `V`, of shape `(m, 1)`, which divides `V` row by row
- `np.linalg.norm(u)`: the norm of `u`
- with the cosine, normalize `u` and each row of `V`, then compute their dot products

On a small example, with $u = (1, 0)$ and two items $v_1 = (2, 0)$ and $v_2 = (1, 1)$: the dot product gives 2 and 1, the cosine 1 and about 0.707 ($\cos 45° = 1/\sqrt 2$).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def compute_scores(query_embedding, item_embeddings, measure=DOT):
  u = query_embedding
  V = item_embeddings
  if measure == COSINE:
    V = V / np.linalg.norm(..., axis=1, keepdims=True)
    u = u / ...
  return u.dot(...)


print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), DOT))
print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), COSINE))
```

#### Solution

In [ ]:
DOT = "dot"
COSINE = "cosine"


def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  u = query_embedding
  V = item_embeddings
  if measure == COSINE:
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    u = u / np.linalg.norm(u)
  scores = u.dot(V.T)
  return scores


print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), DOT))
print(compute_scores(np.array([1.0, 0.0]), np.array([[2.0, 0.0], [1.0, 1.0]]), COSINE))

### Recommendations and nearest neighbors

Equipped with this function, we can compute recommendations, where the query embedding can be either a user embedding or a movie embedding.

In [ ]:
# @title User recommendations and nearest neighbors (run this cell)
def user_recommendations(model, measure=DOT, exclude_rated=False, k=6):
  scores = compute_scores(
    model.embeddings["user_id"][943], model.embeddings["movie_id"], measure
  )
  score_key = measure + " score"
  df = pd.DataFrame(
    {
      score_key: list(scores),
      "movie_id": movies["movie_id"],
      "titles": movies["title"],
      "genres": movies["all_genres"],
    }
  )
  if exclude_rated:
    # remove movies that are already rated
    rated_movies = ratings[ratings.user_id == "943"]["movie_id"].values
    df = df[df.movie_id.apply(lambda movie_id: movie_id not in rated_movies)]
  display.display(df.sort_values([score_key], ascending=False).head(k))


def movie_neighbors(model, title_substring, measure=DOT, k=6):
  # Search for movie ids that match the given substring.
  ids = movies[movies["title"].str.contains(title_substring)].index.values
  titles = movies.iloc[ids]["title"].values
  if len(titles) == 0:
    raise ValueError("Found no movies with title %s" % title_substring)
  print("Nearest neighbors of : %s." % titles[0])
  if len(titles) > 1:
    print(
      "[Found more than one matching movie. Other candidates: {}]".format(
        ", ".join(titles[1:])
      )
    )
  movie_id = ids[0]
  scores = compute_scores(
    model.embeddings["movie_id"][movie_id], model.embeddings["movie_id"], measure
  )
  score_key = measure + " score"
  df = pd.DataFrame(
    {score_key: list(scores), "titles": movies["title"], "genres": movies["all_genres"]}
  )
  display.display(df.sort_values([score_key], ascending=False).head(k))

### Your recommendations

The next cell computes the recommendations of user 943, the one of exercise 3: yours, if you entered your ratings there.

In [ ]:
user_recommendations(model, measure=COSINE, k=5)

How do the recommendations look?

### Movie Nearest neighbors

Let's look at the neareast neighbors for some of the movies.

In [ ]:
movie_neighbors(model, "Aladdin", DOT)
movie_neighbors(model, "Aladdin", COSINE)

It seems that the quality of learned embeddings may not be very good. This will be addressed in Section V by adding several regularization techniques. First, we will further inspect the embeddings.

## Movie Embedding Norm

We can also observe that the recommendations with dot-product and cosine are different: with dot-product, the model tends to recommend popular movies. This can be explained by the fact that in matrix factorization models, the norm of the embedding is often correlated with popularity (popular movies have a larger norm), which makes it more likely to recommend more popular items. We can confirm this hypothesis by sorting the movies by their embedding norm, as done in the next cell.

In [ ]:
# @title Embedding Visualization code (run this cell)


def movie_embedding_norm(models):
  """Visualizes the norm and number of ratings of the movie embeddings.
  Args:
    model: A MFModel object.
  """
  if not isinstance(models, list):
    models = [models]
  df = pd.DataFrame(
    {
      "title": movies["title"],
      "genre": movies["genre"],
      "num_ratings": movies_ratings["rating count"],
    }
  )
  charts = []
  brush = alt.selection_interval()
  for i, model in enumerate(models):
    norm_key = "norm" + str(i)
    df[norm_key] = np.linalg.norm(model.embeddings["movie_id"], axis=1)
    nearest = alt.selection_point(
      encodings=["x", "y"],
      on="mouseover",
      nearest=True,
      empty=False,
      name="nearest%d" % i,
    )
    base = (
      alt.Chart()
      .mark_circle()
      .encode(
        x="num_ratings",
        y=norm_key,
        color=alt.condition(brush, alt.value("#4c78a8"), alt.value("lightgray")),
      )
      .add_params(nearest)
    )
    if i == 0:
      base = base.add_params(brush)
    text = (
      alt.Chart()
      .mark_text(align="center", dx=5, dy=-5)
      .encode(
        x="num_ratings", y=norm_key, text=alt.condition(nearest, "title", alt.value(""))
      )
    )
    charts.append(alt.layer(base, text))
  return alt.hconcat(*charts, data=df)


def visualize_movie_embeddings(data, x, y):
  nearest = alt.selection_point(
    encodings=["x", "y"], on="mouseover", nearest=True, empty=False
  )
  base = (
    alt.Chart()
    .mark_circle()
    .encode(
      x=x,
      y=y,
      color=alt.condition(genre_filter, "genre", alt.value("whitesmoke")),
    )
    .properties(width=600, height=600)
    .add_params(nearest)
  )
  text = (
    alt.Chart()
    .mark_text(align="left", dx=5, dy=-5)
    .encode(x=x, y=y, text=alt.condition(nearest, "title", alt.value("")))
  )
  return alt.hconcat(alt.layer(base, text), genre_chart, data=data)


def tsne_movie_embeddings(model):
  """Visualizes the movie embeddings, projected using t-SNE with Cosine measure.
  Args:
    model: A MFModel object.
  """
  tsne = sklearn.manifold.TSNE(
    n_components=2,
    perplexity=40,
    metric="cosine",
    early_exaggeration=10.0,
    init="pca",
    verbose=True,
    max_iter=400,
  )

  print("Running t-SNE...")
  V_proj = tsne.fit_transform(model.embeddings["movie_id"])
  movies.loc[:, "x"] = V_proj[:, 0]
  movies.loc[:, "y"] = V_proj[:, 1]
  return visualize_movie_embeddings(movies, "x", "y")

In [ ]:
movie_embedding_norm(model)

Note: Depending on how the model is initialized, you may observe that some niche movies (ones with few ratings) have a high norm, leading to spurious recommendations. This can happen if the embedding of that movie happens to be initialized with a high norm. Then, because the movie has few ratings, it is infrequently updated, and can keep its high norm. This will be alleviated by using regularization.

### Exercise 6: The initialization's standard deviation

Build and train a model `model_lowinit` like `model`, but with an initialization standard deviation `init_stddev` ten times smaller, 0.05. Then compare its nearest neighbors of "Aladdin" (`movie_neighbors`) and the norms of its embeddings (`movie_embedding_norm`) with those of `model`. One quantity that can be helpful is that the expected norm of a $d$-dimensional vector with entries $\sim \mathcal N(0, \sigma^2)$ is approximately $\sigma \sqrt d$.

How does this affect the embedding norm distribution, and the ranking of the top-norm movies?

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `build_model(ratings, embedding_dim=30, init_stddev=0.05)`, then `train(num_iterations=1000, learning_rate=10.0)`: the parameters of `model`, except the standard deviation
- `movie_neighbors(model_lowinit, "Aladdin", DOT)`, then with `COSINE`
- `movie_embedding_norm([model, model_lowinit])`: the two charts side by side

In dimension 30, the initial norm of an embedding is about $0.5 \sqrt{30} \approx 2.7$ with `model`, and $0.05 \sqrt{30} \approx 0.27$ with `model_lowinit`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model_lowinit = build_model(ratings, embedding_dim=30, init_stddev=...)
model_lowinit.train(num_iterations=1000, learning_rate=10.0)
movie_neighbors(model_lowinit, "Aladdin", DOT)
movie_neighbors(model_lowinit, "Aladdin", COSINE)
movie_embedding_norm([model, ...])
```

#### Solution

In [ ]:
model_lowinit = build_model(ratings, embedding_dim=30, init_stddev=0.05)
model_lowinit.train(num_iterations=1000, learning_rate=10.0)
movie_neighbors(model_lowinit, "Aladdin", DOT)
movie_neighbors(model_lowinit, "Aladdin", COSINE)
movie_embedding_norm([model, model_lowinit])

## Embedding visualization
Since it is hard to visualize embeddings in a higher-dimensional space (when the embedding dimension $k > 3$), one approach is to project the embeddings to a lower dimensional space. T-SNE (T-distributed Stochastic Neighbor Embedding) is an algorithm that projects the embeddings while attempting to preserve their pariwise distances. It can be useful for visualization, but one should use it with care. For more information on using t-SNE, see [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/).

In [ ]:
tsne_movie_embeddings(model_lowinit)

You can highlight the embeddings of a given genre by clicking on the genres panel (SHIFT+click to select multiple genres).

We can observe that the embeddings do not seem to have any notable structure, and the embeddings of a given genre are located all over the embedding space. This confirms the poor quality of the learned embeddings. One of the main reasons, which we will address in the next section, is that we only trained the model on observed pairs, and without regularization.

# V. Regularization In Matrix Factorization

In the previous section, our loss was defined as the mean squared error on the observed part of the rating matrix. This is a problem: the model is never penalized on the unobserved pairs, so it can bring together the embeddings of unrelated movies (that the same users never rated together), which then wrongly get high scores. This phenomenon is known as *folding*.

We will add regularization terms that will address this issue. We will use two types of regularization:
- Regularization of the model parameters. This is a common $\ell_2$ regularization term on the embedding matrices, given by $r(U, V) =  \frac{1}{n} \sum_i \|u_i\|^2 + \frac{1}{m}\sum_j \|v_j\|^2$.
- A global prior that pushes the prediction of any pair towards zero, called the *gravity* term. This is given by $g(U, V) = \frac{1}{nm} \sum_{i = 1}^n \sum_{j = 1}^m (u_i \cdot v_j)^2$.

The total loss is then given by
$$
\frac{1}{|\Omega|}\sum_{(i, j) \in \Omega} (M_{ij} - u_i \cdot v_j)^2 + \lambda _r r(U, V) + \lambda_g g(U, V)
$$
where $\lambda_r$ and $\lambda_g$ are two regularization coefficients (hyper-parameters).

### Exercise 7: Build a regularized Matrix Factorization model and train it
Complete the `build_regularized_model` function, which builds a regularized model: the training and test errors on the observed ratings (`error_train`, `error_test`, with your `sparse_mean_square_error` function), the gravity term multiplied by $\lambda_g$ (`gravity_loss`) and the $\ell_2$ regularization term multiplied by $\lambda_r$ (`regularization_loss`). You are given a function `gravity(U, V)` that computes the gravity term given the two embedding matrices $U$ and $V$.

**Useful documentation**

- [`tf.math.reduce_sum`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_sum)

In [ ]:
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return (
    1.0
    / (U.shape[0].value * V.shape[0].value)
    * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True)
    )
  )


def build_regularized_model(
  ratings, embedding_dim=3, regularization_coeff=0.1, gravity_coeff=1.0, init_stddev=0.1
):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Split the ratings DataFrame into train and test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # SparseTensor representation of the train and test datasets.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )

  # Your code here: error_train, error_test, gravity_loss, regularization_loss
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
    "train_error_observed": error_train,
    "test_error_observed": error_test,
  }
  loss_components = {
    "observed_loss": error_train,
    "regularization_loss": regularization_loss,
    "gravity_loss": gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

#### Additional hints

The calls and parameters to use:

- `sparse_mean_square_error(A_train, U, V)`: the training error; with `A_test`, the test error
- `gravity_coeff * gravity(U, V)`: the gravity term, multiplied by its coefficient
- `tf.reduce_sum(U * U) / U.shape[0].value`: $\frac{1}{n} \sum_i \|u_i\|^2$, the sum of the squared coefficients of $U$ divided by its number of rows; same for $V$
- `regularization_coeff`: the coefficient $\lambda_r$ of the sum of both

A skeleton of the four lines to write instead of `# Your code here`, indented like the rest of the function, replacing each `...`:

```python
error_train = sparse_mean_square_error(..., U, V)
error_test = ...
gravity_loss = gravity_coeff * ...
regularization_loss = regularization_coeff * (
  tf.reduce_sum(U * U) / U.shape[0].value + ...
)
```

#### Solution

In [ ]:
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return (
    1.0
    / (U.shape[0].value * V.shape[0].value)
    * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True)
    )
  )


def build_regularized_model(
  ratings, embedding_dim=3, regularization_coeff=0.1, gravity_coeff=1.0, init_stddev=0.1
):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Split the ratings DataFrame into train and test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # SparseTensor representation of the train and test datasets.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(
    tf.random_normal([A_train.dense_shape[0], embedding_dim], stddev=init_stddev)
  )
  V = tf.Variable(
    tf.random_normal([A_train.dense_shape[1], embedding_dim], stddev=init_stddev)
  )

  error_train = sparse_mean_square_error(A_train, U, V)
  error_test = sparse_mean_square_error(A_test, U, V)
  gravity_loss = gravity_coeff * gravity(U, V)
  regularization_loss = regularization_coeff * (
    tf.reduce_sum(U * U) / U.shape[0].value + tf.reduce_sum(V * V) / V.shape[0].value
  )
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
    "train_error_observed": error_train,
    "test_error_observed": error_test,
  }
  loss_components = {
    "observed_loss": error_train,
    "regularization_loss": regularization_loss,
    "gravity_loss": gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

### Training the regularized model

It is now time to train the regularized model! You can try different values of the regularization coefficients, and different embedding dimensions.

In [ ]:
reg_model = build_regularized_model(
  ratings,
  regularization_coeff=0.1,
  gravity_coeff=1.0,
  embedding_dim=35,
  init_stddev=0.05,
)
reg_model.train(num_iterations=2000, learning_rate=20.0)

Observe that adding the regularization terms results in a higher MSE, both on the training and test set. However, as we will see, the quality of the recommendations improves. This highlights a tension between fitting the observed data and minimizing the regularization terms. Fitting the observed data often emphasizes learning high similarity (between items with many interactions), but a good embedding representation also requires learning low similarity (between items with few or no interactions).

### Inspect the results
Let's see if the results with regularization look better.

In [ ]:
user_recommendations(reg_model, DOT, exclude_rated=True, k=10)

Hopefully, these recommendations look better. You can change the similarity measure from COSINE to DOT and observe how this affects the recommendations.

Since the model is likely to recommend items that you rated highly, you have the option to exclude the items you rated, using `exclude_rated=True`.

In the following cells, we display the nearest neighbors, the embedding norms, and the t-SNE projection of the movie embeddings.

In [ ]:
movie_neighbors(reg_model, "Aladdin", DOT)
movie_neighbors(reg_model, "Aladdin", COSINE)

Here we compare the embedding norms of `model`, `model_lowinit` and `reg_model`. Selecting an area of the first chart highlights its movies on all the charts.

In [ ]:
movie_embedding_norm([model, model_lowinit, reg_model])

In [ ]:
# Visualize the embeddings
tsne_movie_embeddings(reg_model)

We should observe that the embeddings have a lot more structure than the unregularized case. Try selecting different genres and observe how they tend to form clusters (for example Horror, Animation and Children).

### Conclusion
This concludes this section on matrix factorization models. Note that while the scale of the problem is small enough to allow efficient training using gradient descent, many practical problems need to be trained using more specialized algorithms such as Alternating Least Squares (ALS: with $V$ fixed, each $u_i$ is the solution of a least squares problem, then with $U$ fixed, each $v_j$, and so on), for example [Spark MLlib's ALS](https://spark.apache.org/docs/latest/ml-collaborative-filtering.html) or the [implicit](https://benfred.github.io/implicit/) library.

# VI. Softmax model

In this section, we will train a simple softmax model that predicts whether a given user has rated a movie.

The model will take as input a feature vector $x$ representing the list of movies the user has rated. We start from the ratings DataFrame, which we group by user_id.

In [ ]:
rated_movies = (
  ratings[["user_id", "movie_id"]]
  .groupby("user_id", as_index=False)
  .aggregate(lambda x: list(x))
)
rated_movies.head()

We then create a function that generates an example batch, such that each example contains the following features:
- movie_id: A tensor of strings of the movie ids that the user rated.
- genre: A tensor of strings of the genres of those movies
- year: A tensor of strings of the release year.

In [ ]:
# @title Batch generation code (run this cell)
years_dict = {movie: year for movie, year in zip(movies["movie_id"], movies["year"])}
genres_dict = {
  movie: genres.split("-")
  for movie, genres in zip(movies["movie_id"], movies["all_genres"])
}


def make_batch(ratings, batch_size):
  """Creates a batch of examples.
  Args:
    ratings: A DataFrame of ratings such that examples["movie_id"] is a list of
      movies rated by a user.
    batch_size: The batch size.
  """

  def pad(x, fill):
    return pd.DataFrame.from_dict(x).fillna(fill).values

  movie = []
  year = []
  genre = []
  label = []
  for movie_ids in ratings["movie_id"].values:
    movie.append(movie_ids)
    genre.append([x for movie_id in movie_ids for x in genres_dict[movie_id]])
    year.append([years_dict[movie_id] for movie_id in movie_ids])
    label.append([int(movie_id) for movie_id in movie_ids])
  features = {
    "movie_id": pad(movie, ""),
    "year": pad(year, ""),
    "genre": pad(genre, ""),
    "label": pad(label, -1),
  }
  batch = (
    tf.data.Dataset.from_tensor_slices(features)
    .shuffle(1000)
    .repeat()
    .batch(batch_size)
    .make_one_shot_iterator()
    .get_next()
  )
  return batch


def select_random(x):
  """Selectes a random elements from each row of x."""

  def to_float(x):
    return tf.cast(x, tf.float32)

  def to_int(x):
    return tf.cast(x, tf.int64)

  batch_size = tf.shape(x)[0]
  rn = tf.range(batch_size)
  nnz = to_float(tf.count_nonzero(x >= 0, axis=1))
  rnd = tf.random_uniform([batch_size])
  ids = tf.stack([to_int(rn), to_int(nnz * rnd)], axis=1)
  return to_int(tf.gather_nd(x, ids))

### Loss function
The softmax model is a neural network that maps the input features $x$ to a user embedding $\psi(x) \in \mathbb R^d$, where $d$ is the embedding dimension. This vector is then multiplied by a movie embedding matrix $V \in \mathbb R^{m \times d}$ (where $m$ is the number of movies), and the final output of the model is the softmax of the product, one probability per movie:
$$
\hat p(x) = \text{softmax}(\psi(x) V^\top).
$$
Given a target label $y$, if we denote by $p = 1_y$ a one-hot encoding of this target label, then the loss is the cross-entropy between $\hat p(x)$ and $p$:
$$
-\sum_j p_j \log \hat p(x)_j = -\log \hat p(x)_y = -\left(\psi(x) \cdot v_y - \log \sum_{j = 1}^m e^{\psi(x) \cdot v_j}\right).
$$

### Exercise 8: Write a loss function for the softmax model

Write a function that takes the user embeddings $\psi(x)$ of a batch, the movie embeddings $V$ and the target labels $y$ (one integer movie id per example), and returns the mean cross-entropy over the batch. Write it from the `logits`, the product $\psi(x) V^\top$: their log-softmax gives $\log \hat p(x)$, of which we keep, for each example, the value of the target class, before taking the opposite of the mean. Check it on the small example of the hints.

**Useful documentation**

- [`tf.nn.log_softmax`](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax)
- [`tf.one_hot`](https://www.tensorflow.org/api_docs/python/tf/one_hot)
- [`tf.math.reduce_mean`](https://www.tensorflow.org/api_docs/python/tf/math/reduce_mean)

In [ ]:
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: An integer tensor of shape [batch_size], such that labels[i] is the
      target label (a movie id) for example i.
  Returns:
    The mean cross-entropy loss.
  """
  # Your code here
  ...


# Your code here: the check on the small example of the hints

#### Additional hints

The calls and parameters to use:

- `tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)`: the logits, of shape `[batch_size, num_movies]`
- `tf.nn.log_softmax(logits)`: $\log \hat p(x)$, row by row
- `tf.one_hot(labels, depth=tf.shape(logits)[1])`: the one-hot encoding $1_y$ of each label; multiplied by the log-softmax, then summed over axis 1 (`tf.reduce_sum(..., axis=1)`), it keeps the value of the target class
- `tf.reduce_mean`: the mean over the batch
- `tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=labels)` computes the same loss, example by example: a way to check yours

On a small example of two users and three movies, with $\psi(x) = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$, $V = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix}$ and the labels $0$ and $2$: the logits are $\begin{bmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \end{bmatrix}$, each example has a loss of $-(1 - \log(e + 1 + e)) = \log(1 + 2e) - 1$, and the mean loss is about 0.862.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def softmax_loss(user_embeddings, movie_embeddings, labels):
  logits = tf.matmul(..., ..., transpose_b=True)
  log_probs = tf.nn.log_softmax(...)
  one_hot_labels = tf.one_hot(labels, depth=tf.shape(logits)[1])
  return -tf.reduce_mean(tf.reduce_sum(..., axis=1))


toy_user_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0]])
toy_movie_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
toy_labels = tf.constant([0, 2])
with tf.Session() as sess:
  print(sess.run(softmax_loss(toy_user_embeddings, toy_movie_embeddings, toy_labels)))
```

#### Solution

In [ ]:
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: An integer tensor of shape [batch_size], such that labels[i] is the
      target label (a movie id) for example i.
  Returns:
    The mean cross-entropy loss.
  """
  logits = tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)
  log_probs = tf.nn.log_softmax(logits)
  one_hot_labels = tf.one_hot(labels, depth=tf.shape(logits)[1])
  loss = -tf.reduce_mean(tf.reduce_sum(one_hot_labels * log_probs, axis=1))
  return loss


toy_user_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0]])
toy_movie_embeddings = tf.constant([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
toy_labels = tf.constant([0, 2])
with tf.Session() as sess:
  print(sess.run(softmax_loss(toy_user_embeddings, toy_movie_embeddings, toy_labels)))
  # The same loss, computed by TensorFlow in one call
  toy_logits = tf.matmul(toy_user_embeddings, toy_movie_embeddings, transpose_b=True)
  print(
    sess.run(
      tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
          logits=toy_logits, labels=toy_labels
        )
      )
    )
  )

### Exercise 9: Build a softmax model, train it, and inspect its embeddings

We are now ready to build a softmax CFModel. The architecture of the model is defined in the function `create_network` of the next cell: the input embeddings (movie_id, genre and year) are concatenated to form the input layer, then we have hidden layers with dimensions specified by the `hidden_dims` argument. Finally, the output of the last hidden layer, the user embedding $\psi(x)$, is multiplied by the movie embeddings to obtain the logits. For the target label, we will use a randomly-sampled movie_id from the list of movies the user rated.

The model, from input to output:
1. three features of the user: the movies they rated, their genres and their release years;
2. for each one, a bag-of-words embedding, the mean of the embeddings of its values;
3. their concatenation, then the hidden layers, whose output is $\psi(x)$;
4. the logits $\psi(x) V^\top$, one per movie, then their softmax $\hat p(x)$, compared with the target label by the loss of exercise 8.

Complete the `build_softmax_model` function: the training and test losses, `train_loss` and `test_loss` (using your `softmax_loss` function), and the precision at 10 on the test set, `test_precision_at_10`, the share of the target label among the 10 movies of highest logits, averaged over the test examples (at most 0.1, since an example has a single target). The feature columns the function receives are built further down, by `make_embedding_col`.

**Useful documentation**

- [`tf.compat.v1.metrics.precision_at_k`](https://www.tensorflow.org/api_docs/python/tf/compat/v1/metrics/precision_at_k)

In [ ]:
def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of training examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """

  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Create a bag-of-words embedding for each sparse feature.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Hidden layers.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = (
        tf.get_variable(
          "hidden%d_w_" % i,
          shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(stddev=1.0 / np.sqrt(output_dim)),
        )
        / 10.0
      )
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Train
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
      "input_layer/movie_id_embedding/embedding_weights"
    )

  # Your code here: train_loss, test_loss, test_precision_at_10

  metrics = (
    {"train_loss": train_loss, "test_loss": test_loss},
    {"test_precision_at_10": test_precision_at_10},
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

#### Additional hints

The calls and parameters to use:

- `softmax_loss(train_user_embeddings, movie_embeddings, train_labels)`: the training loss; same for the test, with `test_user_embeddings` and `test_labels`
- `tf.metrics.precision_at_k(labels=test_labels, predictions=..., k=10)`: `predictions` gets the test logits, `tf.matmul(test_user_embeddings, movie_embeddings, transpose_b=True)`
- `precision_at_k` returns a pair `(precision, update_op)`: keep `update_op`, which updates the mean at each evaluation and returns it

A skeleton of the three lines to write instead of `# Your code here`, indented like the rest of the function, replacing each `...`:

```python
train_loss = softmax_loss(...)
test_loss = softmax_loss(...)
_, test_precision_at_10 = tf.metrics.precision_at_k(
  labels=..., predictions=tf.matmul(..., ..., transpose_b=True), k=10
)
```

#### Solution

In [ ]:
def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of training examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """

  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Create a bag-of-words embedding for each sparse feature.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Hidden layers.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = (
        tf.get_variable(
          "hidden%d_w_" % i,
          shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(stddev=1.0 / np.sqrt(output_dim)),
        )
        / 10.0
      )
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Train
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
      "input_layer/movie_id_embedding/embedding_weights"
    )

  test_loss = softmax_loss(test_user_embeddings, movie_embeddings, test_labels)
  train_loss = softmax_loss(train_user_embeddings, movie_embeddings, train_labels)
  _, test_precision_at_10 = tf.metrics.precision_at_k(
    labels=test_labels,
    predictions=tf.matmul(test_user_embeddings, movie_embeddings, transpose_b=True),
    k=10,
  )

  metrics = (
    {"train_loss": train_loss, "test_loss": test_loss},
    {"test_precision_at_10": test_precision_at_10},
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

### Train the Softmax model

We are now ready to train the softmax model. You can set the following hyperparameters:
- learning rate
- number of iterations. Note: you can run `softmax_model.train()` again to continue training the model from its current state.
- input embedding dimensions (the second argument of each `make_embedding_col`, in `embedding_cols`)
- number of hidden layers and size of each layer (the `hidden_dims` argument)

Note: since our input features are string-valued (movie_id, genre, and year), we need to map them to integer ids. This is done using [`tf.feature_column.categorical_column_with_vocabulary_list`](https://www.tensorflow.org/api_docs/python/tf/feature_column/categorical_column_with_vocabulary_list), which takes a vocabulary list specifying all the values the feature can take. Then each id is mapped to an embedding vector using [`tf.feature_column.embedding_column`](https://www.tensorflow.org/api_docs/python/tf/feature_column/embedding_column).

In [ ]:
# Create feature embedding columns
def make_embedding_col(key, embedding_dim):
  categorical_col = tf.feature_column.categorical_column_with_vocabulary_list(
    key=key, vocabulary_list=list(set(movies[key].values)), num_oov_buckets=0
  )
  return tf.feature_column.embedding_column(
    categorical_column=categorical_col,
    dimension=embedding_dim,
    # default initializer: trancated normal with stddev=1/sqrt(dimension)
    combiner="mean",
  )


with tf.Graph().as_default():
  softmax_model = build_softmax_model(
    rated_movies,
    embedding_cols=[
      make_embedding_col("movie_id", 35),
      make_embedding_col("genre", 3),
      make_embedding_col("year", 2),
    ],
    hidden_dims=[35],
  )

softmax_model.train(
  learning_rate=8.0, num_iterations=3000, optimizer=tf.train.AdagradOptimizer
)

### Inspect the embeddings

We can inspect the movie embeddings as we did for the previous models. Note that in this case, the movie embeddings are used at the same time as input embeddings (for the bag of words representation of the user history), and as softmax weights.

In [ ]:
movie_neighbors(softmax_model, "Aladdin", DOT)
movie_neighbors(softmax_model, "Aladdin", COSINE)

In [ ]:
movie_embedding_norm([reg_model, softmax_model])

In [ ]:
tsne_movie_embeddings(softmax_model)

## Congratulations!

You have completed this Colab notebook.

If you would like to further explore these models, we encourage you to try different hyperparameters and observe how this affects the quality of the model and the structure of the embedding space. Here are some suggestions:
- Change the embedding dimension.
- In the softmax model: change the number of hidden layers, and the input features. For example, you can try a model with no hidden layers, and only the movie ids as inputs.
- Using other similarity measures: In this Colab notebook, we used dot product $d(u, v_j) = u \cdot v_j$ and cosine $d(u, v_j) = \frac{u \cdot v_j}{\|u\|\|v_j\|}$, and discussed how the norms of the embeddings affect the recommendations. You can also try other variants which apply a transformation to the norm, for example $d(u, v_j) = \frac{u \cdot v_j}{\|v_j\|^\alpha}$.